In [ ]:
# ======================================================================
# OpenET Dataset Selection Strategy
#
# PURPOSE
#
# This notebook extracts monthly basin-average evapotranspiration (ET)
# for benchmark HUC8 watersheds used in the Water Balance Evaluation
# (WBET) project.
#
# ET is extracted from all available OpenET models plus the OpenET
# Ensemble product.
#
# ----------------------------------------------------------------------
# OPENET DATASETS
#
# Individual OpenET models
#
#   • SSEBop
#   • geeSEBAL
#   • DisALEXI
#   • PT-JPL
#   • SIMS
#   • eeMETRIC
#
# Ensemble product is OpenET Ensemble (Median Absolute Deviation Ensemble)
#
# ----------------------------------------------------------------------
# VERSIONING
#
# OpenET currently provides two monthly production versions:
#
#   v2.0
#   v2.1
#
# The two versions overlap for several years.
#
# Instead of hard-coding a single version, this workflow automatically
# selects the appropriate collection based on the requested year.
#
# Current strategy:
#
#     2000–2014  →  OpenET Version 2.0
#
#     2015–2025  →  OpenET Version 2.1
#
# This approach preserves the complete 2000–2025 study period while
# using the most recent OpenET products where available.
#
# A future sensitivity analysis might need to compare Version 2.0 and Version 2.1
# during their overlapping years (2015–2024) to determine whether the
# newer products introduce meaningful differences in basin-average ET.
#
# ----------------------------------------------------------------------
# BANDS
#
# Individual OpenET models store monthly ET in:  band = "et"
#
# The OpenET Ensemble stores monthly ET in: band = "et_ensemble_mad"
#
# This band represents the OpenET ensemble evapotranspiration estimate
# generated using the Median Absolute Deviation (MAD) ensemble approach.
#
# ----------------------------------------------------------------------
# WHAT IS BEING COMPUTED?
#
# Each OpenET monthly image already represents the TOTAL
# evapotranspiration for that month (millimeters).
#
# Example:
#     January 2020
#         Pixel A = 18.4 mm
#         Pixel B = 21.7 mm
#         Pixel C = 17.9 mm
#
# The Earth Engine reducer computes the MEAN of these pixel values over
# the HUC8 watershed.
#
# So: Monthly HUC8 ET = Mean monthly ET depth (mm)
# This is a spatial average only. It is NOT averaging across time.
#
# ----------------------------------------------------------------------
# WATER-YEAR TOTALS
#
# Monthly basin-average ET values will later be summed to produce
# annual and water-year ET totals.
#
#     Jan ET
#   + Feb ET
#   + Mar ET
#   + ...
#   = Water-Year ET
#
# Because all values are expressed as millimeters of water depth over
# the basin, they are directly comparable with:
#
#     • Basin-average precipitation (mm)
#     • Basin runoff depth (mm)
#
# allowing computation of the Water Balance Evaluation:
#
#     WBET = P − Q
#
# and subsequent comparison with all OpenET models.
#
# ======================================================================

In [ ]:
# Version 2026-07-09 (Detka)

# Install Packages
!pip install geemap -q
!pip install cartopy scipy -q
!pip install pycrs -q
!pip install geopandas -q
!pip install earthengine-api geemap pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Initialize Earth Engine
import geemap
import pycrs
import io
import requests
import time
import geopandas as gpd
import cartopy
import glob
import pandas as pd
import shutil
import getpass
import zipfile
import pprint
import ee
import geemap
import pandas as pd
from pathlib import Path

import ee


EE_PROJECT = "watrs-wbet"

ee.Authenticate()

ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized!")

Earth Engine initialized!


In [ ]:
# Quick EE test for USGS HUC8 access - should print a number
import ee

fc = ee.FeatureCollection("USGS/WBD/2017/HUC08")

print(fc.size().getInfo())

2303


In [ ]:
# ======================================================
# Mount Google Drive
# ======================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Update path as needed
project_dir = Path("/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval")

print(project_dir.exists())

input_dir = project_dir / "data" / "input"

print(input_dir.exists())

print(list(input_dir.iterdir()))

Mounted at /content/drive
True
True
[PosixPath('/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/input/gauge_matches_selected_basins.csv')]


In [ ]:
from pathlib import Path

# ======================================================
# Set Project Directories
# ======================================================

# Uses same directory structure and workflow approach for pulling USGS streamflow data.

project_dir = Path(
    "/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval"
)

input_dir = project_dir / "data" / "input"     # Contains gauge_matches_selected_basins.csv with selected HUC8 basins and associated stations
openet_dir = project_dir / "data" / "openet"
reports_dir = project_dir / "data" / "reports" # Will contain QA reports

# Create folders if they don't exist
openet_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

Read the Selected Basin IDs and associated gauge stations

In [ ]:
from pathlib import Path
import pandas as pd

# Project folders
project_dir = Path("/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval")
input_dir = project_dir / "data" / "input"

# Read the basin-gauge lookup table
gauge_table = pd.read_csv(
    input_dir / "gauge_matches_selected_basins.csv",
    dtype={"HUCID": str}
)

# Rename to match our workflow
gauge_table = gauge_table.rename(columns={"HUCID": "huc8"})

print(f"Loaded {len(gauge_table)} HUC8 basins")
gauge_table.head()

Loaded 117 HUC8 basins


,huc8,gauge_in,gauge_out
0,18100204,USGS-09527700,USGS-10259540
1,18090202,NaN,USGS-10251330
2,18070203,NaN,USGS-11078000
3,18070202,NaN,USGS-11070500
4,18060005,NaN,USGS-11152500


In [ ]:
# Create selected HUC8 Earth Engine FeatureCollection

huc8_list = (
    gauge_table["huc8"]
    .dropna()
    .astype(str)
    .tolist()
)

huc8_fc = ee.FeatureCollection("USGS/WBD/2017/HUC08")

selected_huc8 = huc8_fc.filter(
    ee.Filter.inList("huc8", huc8_list)
)

print("Selected HUC8 basins:", selected_huc8.size().getInfo())

Selected HUC8 basins: 111


In [ ]:
# OpenET model/version dictionary

et_dict = {
    "ssebop": {
        "2.0": {"asset": "projects/openet/assets/ssebop/conus/gridmet/monthly/v2_0", "band": "et"},
        "2.1": {"asset": "projects/openet/assets/ssebop/conus/gridmet/monthly/v2_1", "band": "et"},
    },
    "geesebal": {
        "2.0": {"asset": "projects/openet/assets/geesebal/conus/gridmet/monthly/v2_0", "band": "et"},
        "2.1": {"asset": "projects/openet/assets/geesebal/conus/gridmet/monthly/v2_1", "band": "et"},
    },
    "disalexi": {
        "2.0": {"asset": "projects/openet/assets/disalexi/conus/gridmet/monthly/v2_0", "band": "et"},
        "2.1": {"asset": "projects/openet/assets/disalexi/conus/gridmet/monthly/v2_1", "band": "et"},
    },
    "ptjpl": {
        "2.0": {"asset": "projects/openet/assets/ptjpl/conus/gridmet/monthly/v2_0", "band": "et"},
        "2.1": {"asset": "projects/openet/assets/ptjpl/conus/gridmet/monthly/v2_1", "band": "et"},
    },
    # SIMS model omitted because it does not include wildlands.
    # "sims": {
    #     "2.0": {"asset": "projects/openet/assets/sims/conus/gridmet/monthly/v2_0", "band": "et"},
    #     "2.1": {"asset": "projects/openet/assets/sims/conus/gridmet/monthly/v2_1", "band": "et"},
    # },
    "eemetric": {
        "2.0": {"asset": "projects/openet/assets/eemetric/conus/gridmet/monthly/v2_0", "band": "et"},
        "2.1": {"asset": "projects/openet/assets/eemetric/conus/gridmet/monthly/v2_1", "band": "et"},
    },
    # ENSEMBLE download completed already
    # "ensemble": {
    #     "2.0": {"asset": "projects/openet/assets/ensemble/conus/gridmet/monthly/v2_0", "band": "et_ensemble_mad"},
    #     "2.1": {"asset": "projects/openet/assets/ensemble/conus/gridmet/monthly/v2_1", "band": "et_ensemble_mad"},
    # },
}

SCALE = 30
UNITS = "mm"

In [ ]:
# Define a version selector
def choose_openet_version(year, month):
    """
    Primary dataset rule:
    2000-2014 -> OpenET v2.0
    2015-2025 -> OpenET v2.1
    """
    if year < 2015:
        return "2.0"
    else:
        return "2.1"

In [ ]:
# Define the extractor function for OpenET data
def extract_openet_month(model_name, year, month, version=None):
    """
    Extract monthly mean OpenET ET for all selected HUC8 basins.

    If version is None, the function automatically uses:
      - v2.0 for 2000-2014
      - v2.1 for 2015-2025
    """

    if version is None:
        version = choose_openet_version(year, month)

    info = et_dict[model_name][version]

    start = f"{year}-{month:02d}-01"
    end = (pd.Timestamp(start) + pd.DateOffset(months=1)).strftime("%Y-%m-%d")

    collection = (
        ee.ImageCollection(info["asset"])
        .filterDate(start, end)
        .select(info["band"])
    )

    image = collection.mosaic().rename("et_mm")

    stats = image.reduceRegions(
        collection=selected_huc8,
        reducer=ee.Reducer.mean(),
        scale=SCALE
    )

    df = geemap.ee_to_df(stats)

    if "mean" in df.columns:
        df = df.rename(columns={"mean": "et_mm"})

    df["model"] = model_name
    df["openet_version"] = version
    df["year"] = year
    df["month"] = month
    df["year_month"] = f"{year}-{month:02d}"
    df["units"] = UNITS

    return df

In [ ]:
# ============================================================
# Restartable OpenET output folders
# ============================================================

openet_monthly_raw_dir = openet_dir / "monthly_raw"
openet_monthly_raw_dir.mkdir(parents=True, exist_ok=True)

def make_openet_monthly_filename(model_name, year, month):
    return f"openet_{model_name}_{year}_{month:02d}.csv"

In [ ]:
# ============================================================
# Restartable OpenET extraction: ALL MODELS, 2000-2025
# ============================================================

openet_log = []

for model_name in et_dict.keys():
    for year in range(2000, 2026):
        for month in range(1, 13):

            version = choose_openet_version(year, month)

            out_file = (
                openet_monthly_raw_dir /
                make_openet_monthly_filename(model_name, year, month)
            )

            # Skip if this model-month already exists
            if out_file.exists():
                print(f"Skipping {model_name} {year}-{month:02d} - file exists")

                existing_df = pd.read_csv(out_file)

                openet_log.append({
                    "model": model_name,
                    "openet_version": version,
                    "year": year,
                    "month": month,
                    "year_month": f"{year}-{month:02d}",
                    "status": "existing",
                    "rows": len(existing_df),
                    "file": str(out_file)
                })

                continue

            print(f"Processing {model_name} {year}-{month:02d}")

            try:
                df = extract_openet_month(
                    model_name=model_name,
                    year=year,
                    month=month
                )

                df.to_csv(out_file, index=False)

                openet_log.append({
                    "model": model_name,
                    "openet_version": version,
                    "year": year,
                    "month": month,
                    "year_month": f"{year}-{month:02d}",
                    "status": "ok",
                    "rows": len(df),
                    "file": str(out_file)
                })

            except Exception as e:
                openet_log.append({
                    "model": model_name,
                    "openet_version": version,
                    "year": year,
                    "month": month,
                    "year_month": f"{year}-{month:02d}",
                    "status": "failed",
                    "rows": 0,
                    "file": str(out_file),
                    "error": str(e)
                })

            pd.DataFrame(openet_log).to_csv(
                reports_dir / "openet_extraction_log_checkpoint.csv",
                index=False
            )

Skipping ssebop 2000-01 - file exists
Skipping ssebop 2000-02 - file exists
Skipping ssebop 2000-03 - file exists
Skipping ssebop 2000-04 - file exists
Skipping ssebop 2000-05 - file exists
Skipping ssebop 2000-06 - file exists
Skipping ssebop 2000-07 - file exists
Skipping ssebop 2000-08 - file exists
Skipping ssebop 2000-09 - file exists
Skipping ssebop 2000-10 - file exists
Skipping ssebop 2000-11 - file exists
Skipping ssebop 2000-12 - file exists
Skipping ssebop 2001-01 - file exists
Skipping ssebop 2001-02 - file exists
Skipping ssebop 2001-03 - file exists
Skipping ssebop 2001-04 - file exists
Skipping ssebop 2001-05 - file exists
Skipping ssebop 2001-06 - file exists
Skipping ssebop 2001-07 - file exists
Skipping ssebop 2001-08 - file exists
Skipping ssebop 2001-09 - file exists
Skipping ssebop 2001-10 - file exists
Skipping ssebop 2001-11 - file exists
Skipping ssebop 2001-12 - file exists
Skipping ssebop 2002-01 - file exists
Skipping ssebop 2002-02 - file exists
Skipping sse

KeyboardInterrupt: 

In [ ]:
# Sanity QA Checkpoint
print(openet_monthly_raw_dir)
print(openet_monthly_raw_dir.exists())
print(len(list(openet_monthly_raw_dir.glob("*.csv"))))
print(list(openet_monthly_raw_dir.glob("*.csv"))[:5])

openet_log_df = pd.DataFrame(openet_log)
openet_log_df["status"].value_counts()

openet_log_df.head()

/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/monthly_raw
True
1548
[PosixPath('/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/monthly_raw/openet_geesebal_2019_09.csv'), PosixPath('/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/monthly_raw/openet_geesebal_2019_10.csv'), PosixPath('/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/monthly_raw/openet_geesebal_2019_11.csv'), PosixPath('/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/monthly_raw/openet_geesebal_2019_12.csv'), PosixPath('/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/monthly_raw/openet_geesebal_2020_01.csv')]


,model,openet_version,year,month,year_month,status,rows,file
0,ssebop,2.0,2000,1,2000-01,existing,111,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
1,ssebop,2.0,2000,2,2000-02,existing,111,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
2,ssebop,2.0,2000,3,2000-03,existing,111,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
3,ssebop,2.0,2000,4,2000-04,existing,111,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
4,ssebop,2.0,2000,5,2000-05,existing,111,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...


In [ ]:
# ============================================================
# Combine monthly OpenET files by model
# ============================================================

monthly_files = sorted(openet_monthly_raw_dir.glob("openet_*.csv"))

for model_name in et_dict.keys():

    model_files = [
        f for f in monthly_files
        if f.name.startswith(f"openet_{model_name}_")
    ]

    print(f"{model_name}: {len(model_files)} monthly files")

    if len(model_files) == 0:
        print(f"Skipping {model_name} - no files found")
        continue

    model_monthly = pd.concat(
        [pd.read_csv(f) for f in model_files],
        ignore_index=True
    )

    out_file = openet_dir / f"openet_monthly_huc8_{model_name}_primary_2000_2025.csv"

    model_monthly.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")

    model_monthly[
    ["huc8", "name", "model", "openet_version", "year_month", "et_mm"]
].head()

ssebop: 312 monthly files
Saved: /content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/openet_monthly_huc8_ssebop_primary_2000_2025.csv
geesebal: 312 monthly files
Saved: /content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/openet_monthly_huc8_geesebal_primary_2000_2025.csv
disalexi: 300 monthly files
Saved: /content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/openet_monthly_huc8_disalexi_primary_2000_2025.csv
ptjpl: 312 monthly files
Saved: /content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/openet_monthly_huc8_ptjpl_primary_2000_2025.csv
eemetric: 312 monthly files
Saved: /content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/openet/openet_monthly_huc8_eemetric_primary_2000_2025.csv
